# chemagent — Debugging Notebook

End-to-end pipeline using the consolidated `chemagent_mcp.py` server.

**Preferred workflow (data stays on disk)**
```
load_dataset → featurize_dataset → split_prepared_dataset → build_model_from_split_file
```

## 1. Environment setup

In [1]:
import sys
from pathlib import Path

# Resolve workspace root (works whether cwd is notebooks/ or the repo root)
_ws_root = Path.cwd()
if not (_ws_root / "src").exists():
    _ws_root = _ws_root.parent

_servers_dir = _ws_root / "src" / "chemagent" / "servers"

for _p in [str(_ws_root), str(_servers_dir)]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

print(f"Workspace root : {_ws_root}")
print(f"Servers dir    : {_servers_dir}")

Workspace root : c:\Users\janela\OneDrive - uni-bonn.de\Code\AI-Agent-for-Compound-Selectivity-Prediction-and-Explainability
Servers dir    : c:\Users\janela\OneDrive - uni-bonn.de\Code\AI-Agent-for-Compound-Selectivity-Prediction-and-Explainability\src\chemagent\servers


## 2. Imports

In [2]:
from src.chemagent.servers.chemagent_mcp import (
    # ── Dataset tools ──────────────────────────────────────────
    list_available_datasets,
    list_loaded_datasets,
    list_featurizers,
    load_dataset,
    get_dataset_info,
    featurize_dataset,
    split_prepared_dataset,
    load_split,
    # ── ML tools ───────────────────────────────────────────────
    get_available_algorithms,
    get_recommended_metrics,
    build_model_from_split_file,
    evaluate_classification,
    evaluate_regression,
    predict,
)

## 3. Discover available options

In [3]:
list_available_datasets()

{'datasets': ['chembl_activity_data_O00329_P42336.csv',
  'chembl_activity_data_O00329_P48736.csv',
  'chembl_activity_data_P42336_P48736.csv'],
 'count': 3,
 'directory': 'C:\\Users\\janela\\OneDrive - uni-bonn.de\\Code\\AI-Agent-for-Compound-Selectivity-Prediction-and-Explainability\\data\\datasets'}

In [4]:
list_featurizers()

{'ECFP': {'parameters': {'n_bits': '2048', 'radius': '2', 'sparse': 'False'},
  'description': 'Generate ECFP (Morgan) bit-vector fingerprints from SMILES strings.'},
 'MACCS': {'parameters': {},
  'description': 'Generate 166-bit MACCS structural-key fingerprints from SMILES strings.'},
 'RDKitFP': {'parameters': {'n_bits': '2048',
   'min_path': '1',
   'max_path': '7'},
  'description': 'Generate RDKit topological (path-based) fingerprints.'},
 'AtomPairFP': {'parameters': {'n_bits': '2048'},
  'description': 'Generate atom-pair fingerprints.'},
 'TopologicalTorsionFP': {'parameters': {'n_bits': '2048'},
  'description': 'Generate topological-torsion fingerprints.'}}

In [5]:
get_available_algorithms()

{'RFR': {'name': 'Random Forest Regressor',
  'task_type': 'regression',
  'hyperparameters': {'n_estimators': [50, 100, 200],
   'max_features': ['sqrt', 'log2'],
   'min_samples_split': [2, 5, 10],
   'min_samples_leaf': [1, 2, 4]},
  'supports_multiclass': False,
  'supports_class_weight': False,
  'description': 'Ensemble of decision trees for regression tasks'},
 'RFC': {'name': 'Random Forest Classifier',
  'task_type': 'classification',
  'hyperparameters': {'n_estimators': [50, 100, 200],
   'max_features': ['sqrt', 'log2'],
   'min_samples_split': [2, 5, 10],
   'min_samples_leaf': [1, 2, 4]},
  'supports_multiclass': True,
  'supports_class_weight': True,
  'description': 'Ensemble of decision trees for classification, handles multi-class'},
 'SVC': {'name': 'Support Vector Classifier',
  'task_type': 'classification',
  'hyperparameters': {'C': [0.1, 1, 10],
   'kernel': ['rbf', 'linear'],
   'gamma': ['scale', 'auto']},
  'supports_multiclass': True,
  'supports_class_weigh

In [6]:
get_recommended_metrics()

{'binary_classification': {'optimization': ['f1',
   'roc_auc',
   'average_precision',
   'accuracy'],
  'evaluation': ['MCC',
   'F1',
   'Precision',
   'Recall',
   'AUC',
   'Average Precision',
   'Balanced Accuracy']},
 'binary_imbalanced': {'optimization': ['f1', 'average_precision', 'roc_auc'],
  'evaluation': ['MCC', 'F1', 'Precision', 'Recall', 'Balanced Accuracy'],
  'note': "Use reg_class='classification-cw' for automatic class weighting"},
 'multiclass': {'optimization': ['f1_macro',
   'f1_weighted',
   'balanced_accuracy',
   'accuracy'],
  'evaluation': ['MCC',
   'Balanced Accuracy',
   'F1_macro',
   'F1_weighted',
   'Precision_macro',
   'Recall_macro',
   'Accuracy']},
 'regression': {'optimization': ['neg_mean_squared_error',
   'neg_root_mean_squared_error',
   'neg_mean_absolute_error',
   'r2'],
  'evaluation': ['RMSE', 'MAE', 'MSE', 'R2', 'Pearson r']}}

## 4. Load dataset

In [7]:
dataset_info = load_dataset("data/datasets/chembl_activity_data_O00329_P48736.csv")
dataset_info

{'dataset_id': 'chembl_activity_data_O00329_P48736',
 'n_samples': 1277,
 'columns': ['smiles', 'class_label', 'pPot_diff', 'target_pair', 'cid'],
 'label_col': 'class_label',
 'label_stats': {'mean': 0.28191072826938135,
  'std': 0.5268901818796558,
  'min': 0.0,
  'max': 2.0,
  'unique_values': 3},
 'has_smiles': True,
 'has_precomputed_features': False,
 'loaded': True,
 'next_step': "Call featurize_dataset(dataset_id='chembl_activity_data_O00329_P48736', method='ECFP', radius=2, n_bits=2048) to compute fingerprints server-side, then split_prepared_dataset().",
 'smiles_sample': ['CC(NC(=O)c1c(N)nn2cccnc12)c1cc2cccc(Cl)c2c(=O)n1-c1ccc(O)cc1',
  'Cc1cccc(NS(=O)(=O)c2ccc(C)c(-c3cnc(N)c(-c4cnn(C)c4)c3)c2)n1',
  'Cc1ccc(S(=O)(=O)NCC(C)(C)O)cc1-c1cnc(N)c(-c2ncco2)c1']}

In [8]:
get_dataset_info(dataset_info["dataset_id"])

{'dataset_id': 'chembl_activity_data_O00329_P48736',
 'loaded': True,
 'raw_data': {'n_samples': 1277,
  'columns': ['smiles', 'class_label', 'pPot_diff', 'target_pair', 'cid'],
  'label_col': 'class_label',
  'smiles_col': 'smiles',
  'id_col': None},
 'prepared': False}

## 5. Featurize (server-side — no large arrays transferred)

In [9]:
featurized = featurize_dataset(
    dataset_info["dataset_id"],
    method="MACCS",

)
featurized

{'dataset_id': 'chembl_activity_data_O00329_P48736',
 'method': 'MACCS',
 'n_samples': 1277,
 'n_features': 167,
 'label_stats': {'mean': 0.28191072826938135,
  'std': 0.5268901818796558,
  'min': 0.0,
  'max': 2.0,
  'unique_values': 3},
 'prepared': True,
 'next_step': "Call split_prepared_dataset('chembl_activity_data_O00329_P48736', train_size=0.7, val_size=0.0, test_size=0.3, stratified=True) to create splits."}

## 6. Split

In [10]:
data_splits = split_prepared_dataset(
    dataset_info["dataset_id"],
    train_size=0.7,
    val_size=0.0,
    test_size=0.3,
    stratified=True,
)
data_splits

{'train': {'n_samples': 893,
  'indices': [726,
   238,
   1120,
   135,
   777,
   318,
   467,
   559,
   350,
   480,
   1062,
   830,
   759,
   750,
   896,
   490,
   155,
   213,
   857,
   1058,
   1087,
   489,
   905,
   1175,
   273,
   497,
   758,
   991,
   1131,
   1182,
   74,
   1252,
   237,
   206,
   1090,
   1116,
   973,
   326,
   712,
   666,
   267,
   800,
   24,
   563,
   671,
   582,
   519,
   194,
   399,
   775,
   76,
   152,
   1045,
   521,
   172,
   1267,
   1251,
   418,
   597,
   752,
   585,
   1117,
   952,
   1020,
   968,
   66,
   1204,
   300,
   475,
   1220,
   556,
   913,
   628,
   417,
   552,
   941,
   788,
   107,
   330,
   12,
   1051,
   265,
   879,
   641,
   1233,
   867,
   430,
   1236,
   944,
   16,
   96,
   693,
   776,
   1210,
   419,
   167,
   166,
   159,
   526,
   1043,
   328,
   1018,
   575,
   640,
   289,
   771,
   234,
   203,
   928,
   1163,
   983,
   425,
   471,
   518,
   570,
   689,
   359,
   101,

## 7. Build model (tune → train → evaluate)

In [11]:
model_result = build_model_from_split_file(
    split_file_path=data_splits["saved_to"],
    algorithm="RFC",
    task="classification",
    )
model_result

{'algorithm': 'RFC',
 'task': 'classification',
 'cv_fold': 5,
 'opt_metric': 'balanced_accuracy',
 'best_params': {'max_features': 'log2',
  'min_samples_leaf': 1,
  'min_samples_split': 2,
  'n_estimators': 200},
 'cv_best_score': 0.7932551599218266,
 'model_path': 'C:\\Users\\janela\\OneDrive - uni-bonn.de\\Code\\AI-Agent-for-Compound-Selectivity-Prediction-and-Explainability\\data\\models\\chembl_activity_data_O00329_P48736_random_RFC.pkl',
 'hyperparameters_searched': {'n_estimators': [50, 100, 200],
  'max_features': ['sqrt', 'log2'],
  'min_samples_split': [2, 5, 10],
  'min_samples_leaf': [1, 2, 4]},
 'train_evaluation': {'target': 'train',
  'algorithm': 'RFC',
  'overall_metrics': {'MCC': 0.9883437876728745,
   'Accuracy': 0.9955207166853304,
   'BA': 0.9788103254769922,
   'F1 macro': 0.9827292576978112,
   'F1 weighted': 0.9954963838586987},
  'per_class_metrics': {'Class_0': {'Precision': 0.997037037037037,
    'Recall': 0.997037037037037,
    'F1': 0.997037037037037,
    

## 8. Inspect results

In [12]:
print("Best params  :", model_result["best_params"])
print("CV best score:", model_result["cv_best_score"])
print("Model saved  :", model_result["model_path"])

Best params  : {'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
CV best score: 0.7932551599218266
Model saved  : C:\Users\janela\OneDrive - uni-bonn.de\Code\AI-Agent-for-Compound-Selectivity-Prediction-and-Explainability\data\models\chembl_activity_data_O00329_P48736_random_RFC.pkl


In [13]:
model_result["train_evaluation"]

{'target': 'train',
 'algorithm': 'RFC',
 'overall_metrics': {'MCC': 0.9883437876728745,
  'Accuracy': 0.9955207166853304,
  'BA': 0.9788103254769922,
  'F1 macro': 0.9827292576978112,
  'F1 weighted': 0.9954963838586987},
 'per_class_metrics': {'Class_0': {'Precision': 0.997037037037037,
   'Recall': 0.997037037037037,
   'F1': 0.997037037037037,
   'Support': 675},
  'Class_1': {'Precision': 0.9946236559139785,
   'Recall': 1.0,
   'F1': 0.9973045822102425,
   'Support': 185},
  'Class_2': {'Precision': 0.96875,
   'Recall': 0.9393939393939394,
   'F1': 0.9538461538461539,
   'Support': 33}},
 'confusion_matrix': [[673, 1, 1], [0, 185, 0], [2, 0, 31]],
 'class_labels': [0, 1, 2]}

In [14]:
model_result["test_evaluation"]

{'target': 'test',
 'algorithm': 'RFC',
 'overall_metrics': {'MCC': 0.7814114724810334,
  'Accuracy': 0.9192708333333334,
  'BA': 0.8006741355060866,
  'F1 macro': 0.8482968926566223,
  'F1 weighted': 0.9162998578584048},
 'per_class_metrics': {'Class_0': {'Precision': 0.9218241042345277,
   'Recall': 0.9758620689655172,
   'F1': 0.948073701842546,
   'Support': 290},
  'Class_1': {'Precision': 0.9090909090909091,
   'Recall': 0.759493670886076,
   'F1': 0.8275862068965517,
   'Support': 79},
  'Class_2': {'Precision': 0.9090909090909091,
   'Recall': 0.6666666666666666,
   'F1': 0.7692307692307693,
   'Support': 15}},
 'confusion_matrix': [[283, 6, 1], [19, 60, 0], [5, 0, 10]],
 'class_labels': [0, 1, 2]}

## 9. Standalone evaluation (optional — pass custom predictions)

In [15]:
import joblib, numpy as np

split    = joblib.load(data_splits["saved_to"])
model    = joblib.load(model_result["model_path"])
X_test   = np.array(split["test_features"])
y_test   = split["test_labels"].tolist()

y_pred   = model.predict(X_test).tolist()
y_proba  = model.predict_proba(X_test).tolist()

evaluate_classification(
    labels=y_test,
    predictions=y_pred,
    probabilities=y_proba,
)

{'target': None,
 'algorithm': 'Model',
 'overall_metrics': {'MCC': 0.7814114724810334,
  'Accuracy': 0.9192708333333334,
  'BA': 0.8006741355060866,
  'F1 macro': 0.8482968926566223,
  'F1 weighted': 0.9162998578584048},
 'per_class_metrics': {'Class_0': {'Precision': 0.9218241042345277,
   'Recall': 0.9758620689655172,
   'F1': 0.948073701842546,
   'Support': 290},
  'Class_1': {'Precision': 0.9090909090909091,
   'Recall': 0.759493670886076,
   'F1': 0.8275862068965517,
   'Support': 79},
  'Class_2': {'Precision': 0.9090909090909091,
   'Recall': 0.6666666666666666,
   'F1': 0.7692307692307693,
   'Support': 15}},
 'confusion_matrix': [[283, 6, 1], [19, 60, 0], [5, 0, 10]],
 'class_labels': [0, 1, 2]}

## 10. Scratch / exploratory